
# 20 试得越多，为何越容易挖到假因子

> **本章要解决的问题**：试了 20 个因子，最好的 t 值 2.5，能不能说“显著”？为什么“只报最好的”不可信？Bonferroni、FDR、deflated Sharpe 各自在控制什么？

## 本章导读

第03章讲单因子报告，第16章讲独立报告。这一章讲**多重检验**：

- 单次检验的 5% vs 多次检验的族错误率；
- Bonferroni / FDR；
- 最大值分布 / bootstrap；
- deflated Sharpe；
- 研究流程的防线。

### 学习目标

- 能算多次检验下“至少一个假阳性”的概率。
- 会用 Bonferroni/FDR 校正阈值。
- 能模拟纯噪声下最大 t 值的分布。
- 能解释 deflated Sharpe 的近似门槛。
- 会记录试验次数并写进研究结论。

### 先修

第02章的 p 值；第03章的 IC；会 numpy。

---

### 环境准备



## 本章知识结构图

<svg style="max-width:100%;height:auto" xmlns="http://www.w3.org/2000/svg" width="800" height="450" viewBox="0 0 800 450" role="img" aria-label="20 试得越多，为何越容易挖到假因子" font-family="Microsoft YaHei, SimHei, sans-serif">
<title>20 试得越多，为何越容易挖到假因子｜知识图</title>
<desc>本章主线：试 20 次，至少一个假阳性的概率约 64%；先记试验次数，再谈显著。</desc>
<rect x="0" y="0" width="800" height="450" fill="#FFFFFF"/>
<defs><marker id="a29" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#1B3A5C"/></marker><marker id="ao29" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#E07B39"/></marker><marker id="ag29" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#6E7B8B"/></marker></defs>
<rect x="40" y="19" width="46" height="29" fill="#E07B39" rx="8"/>
<text x="63" y="39" font-size="15" fill="#FFFFFF" text-anchor="middle" font-weight="bold">20</text>
<text x="98" y="41" font-size="22" fill="#1B3A5C" font-weight="bold">试得越多，为何越容易挖到假因子</text>
<line x1="40" y1="57" x2="760" y2="57" stroke="#E6EFF7" stroke-width="2" stroke-linecap="round"/>
<line x1="40" y1="407" x2="760" y2="407" stroke="#E6EFF7" stroke-width="1.6" stroke-linecap="round"/>
<rect x="40" y="418" width="4" height="10" fill="#E07B39" rx="1"/>
<text x="52" y="428" font-size="13" fill="#6E7B8B">本章主线：试 20 次，至少一个假阳性的概率约 64%；先记试验次数，再谈显著。</text>
<rect x="41" y="88" width="160" height="122" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="121" y="133.8" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">试验次数与 t 值</text>
<text x="121" y="153.9" font-size="10.5" fill="#6E7B8B" text-anchor="middle">族、冠军、Sharpe、样本长</text>
<text x="121" y="168.9" font-size="10.5" fill="#6E7B8B" text-anchor="middle">度</text>
<rect x="51" y="97" width="34" height="17" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.1" rx="7"/>
<text x="68" y="109.5" font-size="9.5" fill="#6E7B8B" text-anchor="middle">输入</text>
<rect x="227" y="88" width="160" height="122" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="307" y="132.1" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">Bonferroni / FDR /</text>
<text x="307" y="150.6" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">DSR</text>
<text x="307" y="170.6" font-size="10.5" fill="#6E7B8B" text-anchor="middle">族错误率、最大值分布</text>
<rect x="237" y="97" width="34" height="17" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.1" rx="7"/>
<text x="254" y="109.5" font-size="9.5" fill="#6E7B8B" text-anchor="middle">处理</text>
<rect x="413" y="88" width="160" height="122" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="493" y="141.3" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">校正后的显著性</text>
<text x="493" y="161.4" font-size="10.5" fill="#6E7B8B" text-anchor="middle">阈值与置信区间</text>
<rect x="423" y="97" width="34" height="17" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.1" rx="7"/>
<text x="440" y="109.5" font-size="9.5" fill="#6E7B8B" text-anchor="middle">输出</text>
<rect x="599" y="88" width="160" height="122" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="679" y="141.3" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">样本外与试验记录</text>
<text x="679" y="161.4" font-size="10.5" fill="#6E7B8B" text-anchor="middle">只报最好的 = 不可信</text>
<rect x="609" y="97" width="34" height="17" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.1" rx="7"/>
<text x="626" y="109.5" font-size="9.5" fill="#6E7B8B" text-anchor="middle">检查</text>
<path d="M 201 149 L 227 149" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a29)"/>
<path d="M 387 149 L 413 149" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a29)"/>
<path d="M 573 149 L 599 149" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a29)"/>
<rect x="41" y="232" width="718" height="152" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.5" rx="14"/>
<rect x="58" y="246" width="4" height="11" fill="#E07B39" rx="1"/>
<text x="70" y="256" font-size="11.5" fill="#1B3A5C" font-weight="bold">多重检验下的“显著”</text>
<text x="58" y="276" font-size="9.5" fill="#6E7B8B">20 次试验的 t 值：最大 2.5 也可能只是噪声</text>
<path d="M 90 356 L 720 356" fill="none" stroke="#6E7B8B" stroke-width="1.2" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag29)"/>
<text x="724" y="370" font-size="8.6" fill="#6E7B8B" text-anchor="end">t 值</text>
<line x1="499.2" y1="282" x2="499.2" y2="358" stroke="#E07B39" stroke-width="1.4" stroke-linecap="round" stroke-dasharray="5 3"/>
<text x="503.2" y="292" font-size="8.6" fill="#E07B39">1.96</text>
<circle cx="152" cy="348" r="3.2" fill="#E6EFF7" stroke="#6E7B8B" stroke-width="1.1"/>
<circle cx="379.3" cy="339" r="3.2" fill="#E6EFF7" stroke="#6E7B8B" stroke-width="1.1"/>
<circle cx="276" cy="330" r="3.2" fill="#E6EFF7" stroke="#6E7B8B" stroke-width="1.1"/>
<circle cx="451.7" cy="321" r="3.2" fill="#E6EFF7" stroke="#6E7B8B" stroke-width="1.1"/>
<circle cx="100.3" cy="348" r="3.2" fill="#E6EFF7" stroke="#6E7B8B" stroke-width="1.1"/>
<circle cx="338" cy="339" r="3.2" fill="#E6EFF7" stroke="#6E7B8B" stroke-width="1.1"/>
<circle cx="513.7" cy="330" r="3.2" fill="#E6EFF7" stroke="#6E7B8B" stroke-width="1.1"/>
<circle cx="224.3" cy="321" r="3.2" fill="#E6EFF7" stroke="#6E7B8B" stroke-width="1.1"/>
<circle cx="420.7" cy="348" r="3.2" fill="#E6EFF7" stroke="#6E7B8B" stroke-width="1.1"/>
<circle cx="183" cy="339" r="3.2" fill="#E6EFF7" stroke="#6E7B8B" stroke-width="1.1"/>
<circle cx="389.7" cy="330" r="3.2" fill="#E6EFF7" stroke="#6E7B8B" stroke-width="1.1"/>
<circle cx="245" cy="321" r="3.2" fill="#E6EFF7" stroke="#6E7B8B" stroke-width="1.1"/>
<circle cx="482.7" cy="348" r="3.2" fill="#E6EFF7" stroke="#6E7B8B" stroke-width="1.1"/>
<circle cx="131.3" cy="339" r="3.2" fill="#E6EFF7" stroke="#6E7B8B" stroke-width="1.1"/>
<circle cx="317.3" cy="330" r="3.2" fill="#E6EFF7" stroke="#6E7B8B" stroke-width="1.1"/>
<circle cx="358.7" cy="321" r="3.2" fill="#E6EFF7" stroke="#6E7B8B" stroke-width="1.1"/>
<circle cx="203.7" cy="348" r="3.2" fill="#E6EFF7" stroke="#6E7B8B" stroke-width="1.1"/>
<circle cx="441.3" cy="339" r="3.2" fill="#E6EFF7" stroke="#6E7B8B" stroke-width="1.1"/>
<circle cx="172.7" cy="330" r="3.2" fill="#E6EFF7" stroke="#6E7B8B" stroke-width="1.1"/>
<circle cx="555" cy="300" r="6" fill="#E07B39" stroke="#E07B39" stroke-width="1.4"/>
<text x="555" y="286" font-size="9" fill="#E07B39" text-anchor="middle" font-weight="bold">max 2.5</text>
<text x="58" y="294" font-size="9" fill="#1B3A5C">P(至少一个假阳性) ≈ 64%</text>
<text x="58" y="308" font-size="8.6" fill="#6E7B8B">Bonferroni 阈值 ≈ 0.05/20</text>
<text x="600" y="330" font-size="8.6" fill="#E07B39" font-weight="bold">只报最好 = 不可信</text>
</svg>

> **图**：本章主线：试 20 次，至少一个假阳性的概率约 64%；先记试验次数，再谈显著。


In [1]:

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib 中文字体已设置")


matplotlib 中文字体已设置



## 在量化研究中的位置

- **谁用**：研究员、策略评审、风控。
- **输入**：试验次数、t 值/Sharpe、样本长度。
- **输出**：校正阈值、最大值分布、deflated Sharpe、研究决定。
- **在流程的哪一步**：验证与多重检验。
- **失败模式**：只报最好、用单次 1.96、不记试验次数。
- **最小工作流**：记录试验 → 族错误率 → Bonferroni/FDR → 最大值模拟 → 决定。

---
## 1. 族错误率

做 $m$ 次独立检验，每次 $\alpha=0.05$，至少一个假阳性的概率：

$$1-(1-\alpha)^m.$$

**公式拆解**

- **符号**：$\alpha$ 是单次检验的显著性水平（常用 0.05）；$m$ 是独立检验次数；$1-\alpha$ 是单次不犯假阳性的概率。
- **结构**：$m$ 次都不犯假阳性的概率是 $(1-\alpha)^m$；用 1 减去它，就是“至少一次假阳性”的概率（族错误率）。
- **代入**：$\alpha=0.05,m=20$：$(0.95)^{20}\approx0.358$，族错误率 $1-0.358=0.642$，约 64%。
- **边界**：$m=1$ 时回到 $\alpha$；$m$ 越大越接近 1；检验不独立时公式只是近似，真实族错误率可能更低/更高。
- **用途/误读**：用来解释“为什么试得多会挖到假因子”。它不是“某个因子为假的概率”，也不能替代 Bonferroni/FDR/DSR。

### 代码



In [2]:

for m in [1, 5, 20, 100]:
    print("检验次数", m, "至少一个假阳性", round(1 - (1 - 0.05) ** m, 3))


检验次数 1 至少一个假阳性 0.05
检验次数 5 至少一个假阳性 0.226
检验次数 20 至少一个假阳性 0.642
检验次数 100 至少一个假阳性 0.994



**读数**：20 次约 64%，100 次约 99.4%。**“试得多”本身就会制造显著。**

**常见错误**：

1. 只报最好的 p 值。
2. 不记录试了多少次。
3. 把“没校正”当“稳健”。

---

## 2. Bonferroni 与 FDR

### 2.1 Bonferroni

每个检验阈值 $\alpha/m$。保守，控制族错误率。

### 2.2 FDR

控制“被拒绝里假阳性的比例”，比 Bonferroni 宽松，适合探索性研究。

### 代码



In [3]:

m = 20
print("Bonferroni 阈值", 0.05 / m)
print("原始阈值", 0.05)


Bonferroni 阈值 0.0025
原始阈值 0.05



**读数**：20 个检验，Bonferroni 阈值 0.0025；原来 p=0.01 的“显著”不再显著。

**常见错误**：

1. 校正后仍用 0.05 讲故事。
2. FDR 当 Bonferroni 用。
3. 不报告校正方法。

---

## 3. 最大值分布与 deflated Sharpe

### 3.1 最大值

即使全是噪声，$m$ 个 t 值的最大值也会很大。零分布可以通过模拟/置换得到。

### 3.2 代码：20 个噪声 t 的最大值



In [4]:

import numpy as np

rng = np.random.default_rng(2901)
null = np.abs(rng.normal(size=(10_000, 20)))
max_t = null.max(axis=1)
print("纯噪声下最大 |t| 的 95% 分位", round(np.quantile(max_t, 0.95), 3))


纯噪声下最大 |t| 的 95% 分位 3.01



**读数**：20 个纯噪声 t 的最大值，95% 分位约 2.8；看到 2.5 并不稀奇。

**常见错误**：

1. 用单次 t 临界值 1.96 判断“20 个里最好”。
2. 不模拟最大值分布。
3. 把“最好”当“唯一”。

**停下来想一想**：deflated Sharpe 在修正什么？

<details><summary>对照你的思路</summary>

它把“试了多少次/样本多长/收益分布”考虑进去，修正“最大 Sharpe”的显著性；试得越多、样本越短，门槛越高。核心还是多重检验。

</details>

---

## 4. 案例：20 个因子只报最好

假设 20 个因子全是噪声，每个因子的 t 值服从标准正态。重复 1,000 次实验，记录“20 个里最大的 |t|”。



In [5]:

rng = np.random.default_rng(2902)
max_abs_t = np.abs(rng.normal(size=(1000, 20))).max(axis=1)
print("最大值均值", round(max_abs_t.mean(), 3))
print("95% 分位", round(np.quantile(max_abs_t, 0.95), 3))
print("超过 1.96 的比例", round((max_abs_t > 1.96).mean(), 3))


最大值均值 2.165
95% 分位 3.012
超过 1.96 的比例 0.643



**读数**：纯噪声下，20 个里最大的 |t| 平均约 2.4，95% 分位约 2.9；超过 1.96 的比例约 90%。**“最好那个 t=2.5”在 20 次试验里非常普通。**

**常见错误**：

1. 用 1.96 判断“20 个里最好”。
2. 不报“试了多少个”。
3. 把“最大”当“唯一”。

---

## 5. deflated Sharpe 的近似门槛

若年化 Sharpe 为 $SR$，样本 $T$ 年，试了 $m$ 个策略，最大 Sharpe 的零分布近似：

$$SR_{\max}\approx\sqrt{\frac{2\log m}{T}}.$$

**公式拆解**

- **符号**：$SR_{\max}$ 是 $m$ 个策略里最大 Sharpe 的零分布门槛；$m$ 是试验次数；$T$ 是样本年数；$\log$ 是自然对数。
- **结构**：试验越多（$\log m$ 越大）、样本越短（$T$ 越小），门槛越高；这是“最大值”分布，不是单次 Sharpe 的分布。
- **代入**：$m=20,T=1$：$\sqrt{2\log20}\approx\sqrt{5.99}\approx2.45$；$m=100$：$\sqrt{2\log100}\approx3.03$。
- **边界**：$m=1$ 时门槛为 0（没有选择偏差）；$T\to0$ 时门槛爆炸；$m$ 很大时 $\log m$ 增长很慢，但门槛仍上升。
- **用途/误读**：用来给“最大 Sharpe”设一个粗略门槛；它是近似，不是精确 p 值，也不能替代 DSR/PBO 的完整计算。

$m=20,T=1$：门槛约 2.45；$m=100$：约 3.03。**试得越多、样本越短，门槛越高。**

### 代码：门槛



In [6]:

import numpy as np

for m in [1, 5, 20, 100]:
    print("试验次数", m, "最大 Sharpe 近似门槛（T=1）", round(np.sqrt(2 * np.log(m)), 3))


试验次数 1 最大 Sharpe 近似门槛（T=1） 0.0
试验次数 5 最大 Sharpe 近似门槛（T=1） 1.794
试验次数 20 最大 Sharpe 近似门槛（T=1） 2.448
试验次数 100 最大 Sharpe 近似门槛（T=1） 3.035



**读数**：20 次试验、1 年样本，Sharpe 2.5 只是“门槛附近”；不能说“很强”。

**常见错误**：

1. 用单次 Sharpe 的 1.96 门槛。
2. 不报试验次数/样本长度。
3. 把近似门槛当精确 p 值。

---




## 5. DSR/PBO 的落地

- **DSR**：按试验次数/样本长度校正 Sharpe；
- **PBO**：用 CSCV 估计“冠军过拟合”的概率；
- 实现参考：FactorZen、`holdout-backtest`；
- 位置：Alphalens 初筛之后、fincore 组合之前。

完整工具链见第37章。

---

## 本章小结

### 一张表

| 方法 | 控制 | 特点 |
|---|---|---|
| 单次 p | 单次 | 不适用于多次 |
| Bonferroni | 族错误率 | 保守 |
| FDR | 假阳性比例 | 较宽松 |
| 最大值分布 | 最好统计量 | 模拟/置换 |
| deflated Sharpe | 最大 Sharpe | 考虑试验次数 |

### 六句话

1. 试得越多，越容易“显著”。
2. 20 次约 64% 至少一个假阳性。
3. Bonferroni 阈值 $\alpha/m$。
4. FDR 控制被拒绝里的假阳性比例。
5. 最大值分布比单次临界值更合适。
6. 记录试验次数，是研究底线。

---

## 本章习题

**1.** 20 次独立检验，至少一个假阳性概率约多少？

**2.** Bonferroni 校正 20 次，阈值多少？

**3.** 为什么只报最好的 p 值不可信？

**4.** 20 个纯噪声 t 的最大值 95% 分位约多少？

**5.** deflated Sharpe 修正什么？

<details><summary>参考答案</summary>

**1.** $1-0.95^{20}\approx0.64$。

**2.** $0.05/20=0.0025$。

**3.** 多次比较会制造假阳性；只报最好，忽略选择偏差。

**4.** 约 2.8。

**5.** 试验次数/样本长度/分布对“最大 Sharpe”显著性的影响。

</details>

---

## 参考资料

- Harvey, C. R., Liu, Y. & Zhu, H. *… and the Cross-Section of Expected Returns*。
- López de Prado, M. *Advances in Financial Machine Learning*。
